# Phase 5 & 6 — Evaluation and Deployment

### **Executive Summary**

The project answered the central challenge: it is possible to predict whether a customer will be a Detractor before the NPS survey, using only operational data available at the time of delivery.

All technical targets defined in Phase 1 were met by the selected models: a Random Forest Classifier to predict whether a customer will be a Detractor, and Linear Regression to predict the customer's NPS score. 

**What did the data reveal?** 

Exploratory analysis identified that customer dissatisfaction is concentrated in two main operational factors: number of complaints (complaints_count) and delivery delay days (delivery_delay_days), which together account for nearly 40% of the model's predictive power.

Variables such as order value and geographic region showed no relevant association with NPS — the issue lies in service quality, not customer profile.

**Selected model and results** 

| Model | Metric | Target | Result |
|---|---|---|---|
| Random Forest | Recall | ≥ 75% | 95% ✅ |
| Random Forest | AUC-ROC | ≥ 0.80 | 85% ✅ |
| Random Forest | F1-Score | ≥ 0.70 | 88% ✅ |
| Random Forest | Precision | ≥ 60% | 82% ✅ |

**Expected business impact**

With 95% Recall, the model proactively identifies 95 out of every 100 future Detractors before the NPS survey. 

This enables the CRM team to act proactively — through coupons, personalized outreach, or issue resolution — before the customer gives a negative rating.

The business goal set in Phase 1 — reducing Detractors by 10% within 90 days — is achievable with this model in production, given that preventive actions have proven effectiveness in customer experience programs.

**Recommended next steps**

1. Model deployment: integrate the Random Forest into the operational pipeline to generate scores within 24 hours of delivery.

2. A/B test: compare a group receiving preventive action vs. a control group to measure real-world effectiveness.

3. Monitoring: track Recall and AUC-ROC monthly to detect target drift.

4. Retraining: update the model quarterly with new data to maintain performance.


### **Deployment: Production Demonstration**

The model trained in Phase 4 is saved to disk using joblib —
a Python serialization library. In production, this file
would be loaded directly into an API or internal system
without needing to retrain.


In [7]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Load classification data
X_clf = pd.read_csv('X_train.csv')
y_clf = pd.read_csv('y_train.csv').squeeze()  # nps_detrator (0 or 1)

# Train the Random Forest Classifier — same parameters as Phase 4
model_rf_clf = RandomForestClassifier(
    class_weight='balanced',  # offsets 74/26 class imbalance
    random_state=42           # ensures reproducibility
)
model_rf_clf.fit(X_clf, y_clf)

# Save the model to disk
joblib.dump(model_rf_clf, 'model_rf_classifier.pkl')
print(f"✅ Model saved: {type(model_rf_clf)}")

# Note: the model is retrained here with the same parameters as Phase 4
# because the original object was overwritten by subsequent models in Notebook 04.
# In production, the model would be saved immediately after training.

✅ Model saved: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


### **Production Usage Simulation**

Full flow demonstration: a new customer completes the purchase journey and the system immediately returns whether they will be a Detractor and with what probability.

In [8]:
# Load the saved model — simulates production usage
model_production = joblib.load('model_rf_classifier.pkl')

# Simulate a new customer arriving in the system
new_customer = pd.read_csv('X_test.csv').iloc[[0]]

# Prediction
prediction = model_production.predict(new_customer)
probability = model_production.predict_proba(new_customer)

print(f"Prediction: {'Detractor' if prediction[0] == 1 else 'Non-detractor'}")
print(f"Probability of being a Detractor: {probability[0][1]:.2%}")

Prediction: Detractor
Probability of being a Detractor: 89.00%


The model classified this customer as a Detractor with 89% probability — high confidence. In production, this customer would immediately be flagged for the CRM team.

This is the complete production flow:

1. Customer completes delivery
2. System automatically collects operational data  
3. Model returns the classification and probability in real time
4. CRM team is alerted for customers with probability above the defined threshold